# LLM Router — Combined Pipeline

Stacks the **validated** old ensemble (TF-IDF Ridge + embedding Ridge + two-head LGBM, public LB ~0.470)
with two new utility-regression members — a **ModernBERT encoder** (optionally fed LLM difficulty
features) and an **LLM-GBM** — then selects blend weights + a Model_K fallback margin with an honest
cross-fit CV and a strict gate. Runs from local Colab disk; caches/artifacts sync to Drive.

Order: **setup → Step 1 (LLM features) → Step 2 (combined) → submit per the gate**.
Keep the validated 0.470 file as a private-LB finalist regardless.

In [ ]:
# Setup: mount Drive, copy project + validated artifacts to LOCAL disk (insulates from Drive drops).
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import shutil, sys
from pathlib import Path
SRC = Path('/content/drive/MyDrive/NLP Final Project')
DST = Path('/content/proj'); DST.mkdir(exist_ok=True)

for sub in ['src', 'dataset']:
    dst = DST / sub
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(SRC / sub, dst)

# Qwen3 embeddings (search any Output/*/cache).
emb_dst = DST / 'Output/router_a100/cache'; emb_dst.mkdir(parents=True, exist_ok=True)
found = []
for pat in ['qwen3_4b_dim1024_chunks_v1_*.npy', '*chunks*_train_*.npy', '*chunks*_test_*.npy']:
    found += list((SRC / 'Output').rglob(pat))
for f in sorted(set(found)): shutil.copy(f, emb_dst / f.name)
print('embedding files copied:', len(set(found)))

# Validated artifacts: find whichever Output/* dir holds the saved OOF/test npz and copy it.
marker = 'weighted_ensemble_best_oof_predictions.npz'
hits = sorted((SRC / 'Output').rglob(marker))
if hits:
    srcdir = hits[0].parent
    n_npz = len(list(srcdir.glob('*_predictions.npz')))
    shutil.copytree(srcdir, DST / 'Output' / srcdir.name, dirs_exist_ok=True)
    print('restored validated artifacts from', srcdir.name, f'({n_npz} npz files)')
else:
    print('WARNING: no *_oof_predictions.npz found under Output/ on Drive -> upload the old run dir')

# Prior router_v2 caches (LLM features, encoder per-fold checkpoints) for resume/speed.
v2 = SRC / 'Output/router_v2'
if v2.exists():
    shutil.copytree(v2, DST / 'Output/router_v2', dirs_exist_ok=True)
    print('restored router_v2 cache')

sys.path.insert(0, str(DST))

def sync_outputs():
    shutil.copytree(DST / 'Output/router_v2', SRC / 'Output/router_v2', dirs_exist_ok=True)
    print('synced Output/router_v2 -> Drive')

!pip -q install lightgbm 2>/dev/null
print('setup done; project at', DST)

## Step 1 — LLM difficulty features (transformers backend)
Uses Colab's existing torch (no vLLM / no CUDA matching). Smaller model for speed. Smoke first,
then full; cached to parquet. (To recompute with different settings, delete the parquet first.)

In [ ]:
!pip -q install -U 'transformers>=4.48.0' accelerate 2>/dev/null
from src.config import CFG
from src import data, llm_difficulty as ld
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # smaller = faster for the transformers path

# Smoke (~2-3 min): validates model load + chat template + answer parsing.
cfg_s = CFG(root=DST, smoke=True, llm_id=MODEL, llm_k_samples=2, llm_max_new_tokens=256)
tr_s, te_s, _ = data.load_data(cfg_s)
ld.compute_llm_features(cfg_s, tr_s, 'train', backend='hf')
ld.compute_llm_features(cfg_s, te_s, 'test', backend='hf')
print('LLM smoke OK')

# Full run (caches llm_feats_train_10182.parquet / llm_feats_test_2550.parquet):
cfg = CFG(root=DST, llm_id=MODEL, llm_k_samples=4, llm_max_new_tokens=320, hf_batch_size=16)
train, test, sample = data.load_data(cfg)
ld.compute_llm_features(cfg, train, 'train', backend='hf')
ld.compute_llm_features(cfg, test, 'test', backend='hf')
sync_outputs()

## Step 2 — Combined pipeline (encoder + LLM-GBM, stacked & gated)
Full-data only (it stacks the full-data validated OOF). Trains the utility encoder (~1.5 h, **per-fold
checkpointed and mirrored to Drive** via `drive_cache`, so a disconnect loses at most the in-progress
fold) + the LLM-GBM, then cross-fit CV + gate. If the encoder OOMs, the first epoch errors immediately
-> add `encoder_bs=4` / `encoder_max_len=1024` and rerun (completed folds resume from cache).

In [ ]:
!pip -q install -U 'transformers>=4.48.0' accelerate 2>/dev/null
from src.config import CFG
from src.combine import run_combined
res = run_combined(CFG(root=DST, drive_cache=str(SRC / 'Output/router_v2/cache')))   # mirrors encoder ckpts to Drive; if OOM add encoder_bs=4, encoder_max_len=1024
sync_outputs()

## Step 3 — Submit per the gate
- **Gate PASSED** -> submit `Output/router_v2/submission_combined.csv`.
- **Gate NOT passed** -> keep the validated finalist `submission_candidate_k_fallback_q05.csv` (~0.470) from the artifacts dir.

Report `combined_report.csv` (cv_old_only vs cv_combined, stability) back before submitting.